In [1]:
# Load libraries
import pandas as pd

In [5]:
# Merge the BM, Factor Indicies, and Equities datasets

# 1. Load our datasets 
# Pro-tip: parse_dates ensures Python treats your 'Date' column as actual time objects
df_equity_1 = pd.read_csv('All_Equities_Cleaned.csv', parse_dates=['Dates'])
df_equity_2 = pd.read_csv('Equities_Jabbir.csv', parse_dates=['Dates'])
df_market = pd.read_csv('MSCI_Data.csv', parse_dates=['Date'])

# 2. Set the Date as the index to make alignment easier
df_equity_1.set_index('Dates', inplace=True)
df_equity_2.set_index('Dates', inplace=True)
df_market.set_index('Date', inplace=True)

# 3. Merge them into one master dataframe
# We join on the index (the dates)
df_merged = df_market.join([df_equity_1, df_equity_2], how='inner')

# Take a peek at the result
print(df_merged.head())

            MSCI USA  MSCI USA Value  MSCI USA Low Size  \
1999-01-29   2673.05     3277.734545            2384.21   
1999-02-26   2598.02     3251.106568            2391.18   
1999-03-31   2705.67     3332.419537            2344.66   
1999-04-30   2802.00     3624.972158            2410.24   
1999-05-31   2736.32     3560.597835            2663.08   

            MSCI USA Low Volatility  MSCI USA High Dividend Yield  \
1999-01-29               995.479542                    997.649417   
1999-02-26               969.079546                    970.419053   
1999-03-31               989.782505                    967.408227   
1999-04-30              1033.753199                   1064.030284   
1999-05-31              1024.310711                   1045.278656   

            MSCI USA Quality  MSCI USA Momentum  KATE US Equity  \
1999-01-29        683.831934         512.942863         19.0938   
1999-02-26        658.588413         500.178311         16.8438   
1999-03-31        682.563547 

In [14]:
# Calculate monthly log or arithmetic returns
returns = df_merged.pct_change()

C:\Users\Anas\AppData\Local\Temp\ipykernel_65084\2008826788.py:2: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = df_merged.pct_change()


In [19]:
# Beta Calculation

# 1. Define our targets (Benchmark + Factors)
targets = [
    'MSCI USA',
    'MSCI USA Value', 'MSCI USA Low Size', 'MSCI USA Low Volatility', 
    'MSCI USA High Dividend Yield', 'MSCI USA Quality', 'MSCI USA Momentum'
]

# Identify equity columns (everything in the dataframe that isn't a target or the risk-free rate)
# Make sure to adjust 'RF_Rate' if your column is named differently or already dropped
equities = [col for col in returns.columns if col not in targets]

# 2. Calculate the 60-month rolling variance for ALL targets at once
rolling_vars = returns[targets].rolling(window=60).var()

# 3. Create a dictionary to store the results
all_equities_betas = {}

# 4. Loop through each equity to calculate its specific Betas
for equity in equities:
    # Create an empty DataFrame to store this equity's betas
    equity_betas = pd.DataFrame(index=returns.index)
    
    for target in targets:
        # Calculate rolling covariance between the equity and the target
        rolling_cov = returns[equity].rolling(window=60).cov(returns[target])
        
        # Calculate Beta: Rolling Covariance / Rolling Variance of the target
        equity_betas[f'Beta_{target}'] = rolling_cov / rolling_vars[target]
        
    # Drop the initial 59 months with NaN
    equity_betas.dropna(inplace=True)
    
    # Store the completed DataFrame in our dictionary
    all_equities_betas[equity] = equity_betas

all_equities_betas

In [20]:
all_equities_betas

{'KATE US Equity':             Beta_MSCI USA  Beta_MSCI USA Value  Beta_MSCI USA Low Size  \
 2004-01-30       1.071011             0.981200               -0.174195   
 2004-02-27       1.051220             0.973077               -0.173824   
 2004-03-31       1.079137             0.985706               -0.183136   
 2004-04-30       1.095161             1.048460               -0.178265   
 2004-05-31       1.104481             1.056889               -0.227571   
 ...                   ...                  ...                     ...   
 2019-08-30       0.620214             0.698804                0.163025   
 2019-09-30       0.560739             0.634477                0.239905   
 2019-10-31       0.553807             0.628642                0.259163   
 2019-11-29       0.506752             0.597186                0.213240   
 2019-12-31       0.507241             0.596083                0.212599   
 
             Beta_MSCI USA Low Volatility  Beta_MSCI USA High Dividend Yield  \
